# <font color="blue"> **Graph and Dynamical Systems Approaches to Whole-Brain Neuronal Networks** </font>
## <font color="blue"> **Zebrafish Functional Data** </font>
<figure>
    <img src="https://mcgovern.mit.edu/wp-content/uploads/2021/10/brainmap_900x600.jpg" alt="Connectome" width="300"/>
</figure>


Notebook prepared for the:
<blockquote>

**Frontiers in Neurophotonics Summer School 2026**


2026 June 1 - 12, Quebec City, Canada
</blockquote>


**Author:**
* Patrick Desrosiers ([Dynamica Research Lab](https://dynamicalab.github.io/) & [CERVO Brain Research Center](https://cervo.ulaval.ca/en/))


## <font color="blue"> **Introduction** </font>

Before analyzing network topology or computing structure–function coupling parameters, clean cellular signals must be extracted from raw biophysical data. Whole-brain two-photon calcium imaging yields massive 3D data stacks where functional fluorescent fluctuations are mixed with structural background noise, light scattering, and optical artifacts.

This notebook guides you through **Step 1 of the computational neuroscience pipeline**: transforming raw imaging datasets into an organized functional network graph. Using an AI-assisted programming workflow with Gemini in Google Colab, you will construct a data-processing engine from the ground up:

1. **Interactive Spatial-Temporal Masking:** Develop a multi-dimensional canvas envelope to separate parenchymal tissue from peripheral agarose and ring artifacts.
2. **Local Peak Detection:** Isolate cellular centroids across the larval brain using Gaussian spatial filtering and neighborhood local maxima features.
3. **Biophysical Normalization ($\Delta F / F_0$):** Standardize traces using pre-stimulus resting percentiles (frames $0$–$600$) to insulate the baseline from stimulus-induced activity tails.
4. **Functional Topology Mapping:** Reconstruct synchronous sub-networks by applying temporal Z-score scaling and symmetrical hierarchical clustering to pairwise Pearson correlation matrices.
5. **Geometric Constraint Testing:** Compute the pairwise Euclidean distance matrix between centroids to evaluate how physical space constrains functional interaction, testing an exponential decay fit ($|R(d)| \propto e^{-d/\lambda}$) to determine the characteristic length ($\lambda$) of localized circuits.

By mastering these upstream extraction steps, you will establish the exact mathematical arrays—the Signal Matrix and the Adjacency Matrix—required for downstream graph-theoretical characterization and dynamics modeling.

---

### **Key Computational Steps**

* **Process Calcium Imaging Data:** Segment cells and extract baseline-normalized time series from whole-brain larval zebrafish recordings.
* **Infer Functional Connectivity:** Construct correlation-based networks, apply dynamic range contrast adjustments, and execute symmetrical row/column clustered rasters.
* **Relate Geometry and Topology:** Assess how anatomical distance shapes functional coupling using distance-binned histograms and non-linear exponential model fitting.

## <font color="blue"> **Functional Data** </font>


### **Experimental Setup**

Resonant-scanning two-photon microscopy was used to capture brain-wide neuronal activity at single-cell resolution in transgenic zebrafish larvae (Tg(elavl3:H2B-GCaMP6s)), with **22 larvae** aged 5-7 days post-fertilization (dpf). Larvae were immobilized in low-melting point agarose, and tail movements were tracked by a high-speed infrared camera. During the experiment, larvae were exposed to **static whole-field illumination for 10 minutes**, followed by **abrupt dark transitions (dark flashes) over 3 minutes**.

A piezo-driven objective rapidly acquired nuclear signals across 21 imaging planes, covering the entire larval brain except the most ventral neuronal populations. Imaging volumes were registered to the mapZebrain brain atlas using ANTs registration, transforming the coordinates of all neuronal nuclei into the atlas coordinate system. Neurons were assigned to one of **70 anatomical brain regions**. Five regions were excluded due to inconsistent sampling or absence of labeled nuclei. In total, the activity of approximately 54,464 neurons across 65 brain regions was recorded, representing roughly half of the neuronal population at this developmental stage.

### **Sample Fluorescence Data**

Before we can extract a Functional Connectivity (FC) matrix, raw functional imaging data typically must go through two computationally intensive preprocessing milestones:
1. **Cell Segmentation:** Identifying individual neuronal cell bodies (nuclei) within the high-resolution structural images to isolate distinct regions of interest (ROIs).
2. **Time-Series Extraction:** Tracking the fluctuations of calcium fluorescence ($\Delta F / F$) over time for each isolated cell to map its dynamic activity.

Processing an entire whole-brain volume simultaneously is too heavy for a standard workshop runtime. To keep things lightweight and efficient, we will work with a pre-processed sample dataset from a single optical section (**z-plane #11**) within the z-stack of a **7-day-old larva (Fish 5)**. The data has already been motion-corrected and normalized.

Even a single plane contains a massive amount of data, so the source file has been split into 7 compressed sequential parts (`.tif` files) to ensure stable hosting. They are available in the public repository at:
`https://github.com/pdesrosiers/brain-connectivity-and-dynamics/tree/main/data/zebrafish`

---

#### 💻 **Task:**

> Ask Gemini to write a Python script that programmatically handles this split dataset. The script should:
>
> 1. Download all 7 sequential `.tif` files from the public GitHub repository path.
> 2. Logically combine/concatenate them along the time axis into a single cohesive 3D numpy array (representing *X* pixels $\times$ *Y* pixels $\times$ *Time frames*).
> 3. Generate a summary report printing out the **final shape of the combined array**, its **data type**, and the **total number of frames loaded**.

---



---

#### 💻 **Task:**
>Ask Gemini to write a Python script using numpy and matplotlib.pyplot to generate and display these two projections side-by-side.

>The script should:

>- Compute the Maximum Intensity Projection along the frame axis (axis 0).

>- Compute the Mean Intensity Projection along the frame axis (axis 0).

>- Plot both images side-by-side using a clean colormap (like 'gray', 'viridis', or 'magma') and add appropriate titles to distinguish them.

---

## <font color="blue"> **Interactive ROI Masking: HTML5 Canvas Interface** </font>



While automated spatial-temporal math is highly effective, complex imaging artifacts occasionally require direct human intervention. In standard laboratory settings, researchers frequently draw an explicit **Region of Interest (ROI)** to restrict their downstream pipelines to valid anatomical structures.

To achieve this in Google Colab without relying on finicky Matplotlib GUI toolkits, we can construct an interactive HTML5 `<canvas>` element directly inside our output cell. Using a custom JavaScript snippet, we can listen for mouse clicks, draw a boundary vector overlay in the browser, and communicate those raw coordinates straight back into our Python kernel.

---

#### 💻 **Task:**

Use Gemini to generate an interactive HTML5 drawing canvas script inside Colab that passes mouse-click vertices back to a Python list named `manual_roi_vertices`.

To ensure the script functions perfectly within Google Colab, copy and paste the detailed technical blueprint below into your prompt:

```text
Write a Google Colab Python script that creates an interactive HTML5 canvas interface to draw a polygon over a reference 2D image array named `mean_proj`.

The script must fulfill the following architectural requirements:
1. Image Preparation: Normalize `mean_proj` to an 8-bit scale (0-255), convert it to a grayscale PIL Image, encode it into a base64 string, and inject it as a background image for an HTML5 canvas element.
2. Web UI Elements: Create a `<canvas>` element with dimensions matching `mean_proj.shape`. Include a red "Clear Points" button and a green "Lock Mask & Save" button.
3. Front-End Drawing Engine (JavaScript): Add event listeners to capture sequential click coordinates $(x, y)$. Draw a green stroke (`#00ff00`) connecting the vertices and place small red anchor dots at each point.
4. Python-Kernel Linkage: Use `google.colab.output.register_callback` to establish a bridge called 'notebook.save_roi_points'. When the save button is clicked, invoke this bridge to send the JavaScript point array back into a Python function.
5. Storage: The callback function must parse the coordinates and store them as a list of $(x, y)$ tuples inside a global variable named `manual_roi_vertices`.

### **Constructing the Binary Array Mask and Verifying Spatial Pruning**

Now that your boundary coordinates have successfully crossed the bridge from the web browser back into the Python kernel, they exist as a collection of vector vertices in the global list `manual_roi_vertices`. However, to isolate our neural signals, we need to convert these sparse coordinate points into a discrete **binary pixel mask**.

We achieve this by utilizing the polygon filling algorithms in `scikit-image`. This process maps our vector coordinates onto the pixel grid of `mean_proj`, determining exactly which pixels fall *inside* your boundary (assigned a value of `True`) and which fall *outside* in the peripheral agarose or ring artifacts (assigned a value of `False`).

---

#### 💻 **Task:**

Use Gemini to write a Python script that transforms your vector coordinates into a binary mask and plots a side-by-side diagnostic validation figure.

To ensure the script functions perfectly within Google Colab, copy and paste the detailed technical blueprint below into your prompt:

```text
Write a Google Colab Python script to convert drawn polygon vertices into a binary mask array.

The script must fulfill the following architectural requirements:
1. Standard Plotting: Include `%matplotlib inline` at the absolute top of the cell to reset the plotting backend.
2. Safety Guardrail: Wrap the execution in an `if/else` statement that checks if `manual_roi_vertices` exists in the global workspace and is not empty. If empty, print an explicit error message instructing the user to lock their canvas first.
3. Mask Synthesis: Initialize a boolean array of zeros named `manual_brain_mask` matching the shape of `mean_proj`. Use `skimage.draw.polygon` to find all row/column pixel coordinates enclosed by the vertices, and set those indices to `True`.
4. Console Report: Print a structured summary showing the total count of enclosed pixels and the exact percentage of the frame area that remains active.
5. Symmetrical Diagnostic Figure: Generate a 1x2 Matplotlib subplot layout (figsize=14x7).
   - Subplot 1: Display `mean_proj` in grayscale with your user-defined vector boundary overlaid as a closed lime-green line (ensure you append the first vertex to the end of your arrays to close the loop).
   - Subplot 2: Create a copy of `mean_proj`, set all pixels outside the mask (`~manual_brain_mask`) to zero, and display this pruned preview using the 'magma' colormap.
6. Figure Formatting: Turn off all axis ticks (`axis('off')`) on both subplots to create a clean, publication-ready visual verification.

## <font color="blue"> **Cell Segmentation and Signal Extraction** </font>



With our manual brain mask established, we can execute the definitive step of the raw processing pipeline: locating individual neurons and extracting their functional time series.

This process requires a multi-stage approach. First, we smooth the structural image to reduce pixel-level noise and extract local intensity peaks as potential neural coordinates using `scikit-image`. Next, we discard any peaks found outside our hand-drawn brain envelope. Finally, we convert these spatial coordinates into functional $\Delta F / F_0$ traces using a baseline normalization strategy tailored to our experimental timeline (the 10-minute quiet resting window), and apply a standard-deviation quality control filter to eliminate dead or inactive pixels.

---

### 💻 **Task:**

Use Gemini to write a high-performance Python script that extracts cell coordinates, normalizes their fluorescence traces, filters inactive cells, and plots a dual-panel verification dashboard.

To ensure the script functions perfectly within Google Colab, copy and paste the detailed technical blueprint below into your prompt:

```text
Write a Google Colab Python script to perform peak extraction, baseline normalization, and signal filtering on a whole-brain 3D movie array named `combined_data`.

The script must fulfill the following architectural requirements:
1. Spatial Filtering & Segmentation: Smooth the reference 2D array `mean_proj` using `skimage.filters.gaussian` with a sigma of 1.0. Locate local peak coordinates using `skimage.feature.peak_local_max` with a minimum distance of 3 pixels, an absolute threshold set to the 55th percentile of the smoothed image, and `exclude_border=True`.
2. Geometric Slicing: Mask the resulting peak array against your `manual_brain_mask`. Keep only the $(x, y)$ coordinate pairs that fall inside the true tissue boundaries.
3. Vectorized Time-Series Extraction: Use the filtered spatial coordinates to index directly into the 3D `combined_data` array along its spatial dimensions, extracting a raw matrix of shape (Cells, Frames).
4. Biophysical Normalization ($\Delta F / F_0$): Calculate the baseline fluorescence vector $F_0$ using a conservative 15th percentile computed strictly across the pre-stimulus resting window (frames 0 to 600). Compute the $\Delta F / F_0$ matrix relative to this baseline, stabilizing the denominator with `np.clip(F0_resting, 1.0, None)`.
5. Signal Quality Control: Calculate the standard deviation of each cell's trace across the full session. Create a boolean mask to keep only cells with a standard deviation greater than 0.03, effectively pruning dead or completely flat pixels. Save the final outputs as `clean_dff`, `x_spatial`, and `y_spatial`.
6. Advanced Diagnostic Layout: Construct an 18x10 Matplotlib figure using `plt.GridSpec` to form a 2x2 multi-panel canvas layout:
   - Panel 1 (Left column, spanning both rows): Display `mean_proj` in grayscale with your active cell coordinates overlaid as a high-density green scatter plot (marker size=4).
   - Panel 2 (Top right): Randomly sample 3 active cells and plot a zoomed-in view of their traces during the sensory transition phase (frames 600 to 750) to visually verify clean calcium transients and exponential decay tails.
   - Panel 3 (Bottom right): Plot the full-session timeline (all 892 frames) for those same 3 sample cells, adding a distinct red vertical dashed line at frame 600 to mark the onset of the dark flashes.

### **Population Visualization: Activity Heatmaps and Clustermaps**

Extracting individual cell coordinates and verifying single traces is a crucial sanity check, but the power of whole-brain imaging lies in analyzing the **collective activity of the entire neural population**. To view the global dynamics of the larval brain all at once, we transition from individual line plots to a population **raster heatmap**.

In this representation, the functional matrix is displayed as a 2D image where every row corresponds to a single neuron, the horizontal axis represents time frames, and the color intensity reflects the magnitude of the $\Delta F / F_0$ signal. However, if the rows are arranged randomly, the global structure will look like disorganized noise. To organize this space, we introduce **Hierarchical Clustering**. By grouping rows with similar temporal profiles together, the algorithm reorders our population matrix, causing coordinated neural assemblies (or ensembles) to emerge as distinct, synchronized horizontal bands.

---

#### 💻 **Task:**

Use Gemini to write a Python script that renders a standard population heatmap side-by-side with an organized hierarchical clustermap of the active neural traces.

To ensure the script functions perfectly within Google Colab, copy and paste the detailed technical blueprint below into your prompt:

```text
Write a Google Colab Python script to visualize population-level neural activity using Seaborn and Matplotlib heatmaps.

The script must fulfill the following architectural requirements:
1. Figure 1 (Standard Population Raster): Generate a standard Matplotlib figure (figsize=12x8) displaying the raw `clean_dff` matrix using `plt.imshow` (or `sns.heatmap`). Set the colormap to 'magma' with color limits (`vmin=0`, `vmax=2.0`). Label the horizontal axis as "Time (Frames)" and the vertical axis as "Neurons (Unordered Index)". Add a red vertical dashed line at frame 600 to mark the dark flash transition.
2. Figure 2 (Hierarchical Clustermap): Use `sns.clustermap` to run a hierarchical clustering analysis on the population matrix.
   - Enforce `row_cluster=True` to group neurons by functional profile similarity.
   - Set `col_cluster=False` to preserve the chronological experimental timeline intact along the horizontal axis.
   - Use 'euclidean' distance as the metric and 'ward' linkage as the clustering method.
   - Use the 'magma' colormap, cap the intensity thresholds between `vmin=0` and `vmax=2.0` for visual consistency, and turn off the dense vertical row text labels (`yticklabels=False`).
   - Add a cyan vertical dashed line at frame 600 across the heatmap axis (`g.ax_heatmap.axvline`) to mark the onset of the dark flashes.

### **Exposing Hidden Dynamics: Temporal Z-Score Standardization**

While a raw $\Delta F / F_0$ heatmap provides a global view of population activity, absolute amplitude variations can severely mask underlying network dynamics. A neuron displaying an exceptionally large calcium influx will statistically overwhelm a neighboring cell that is firing in perfect synchrony but at a lower fluorescent intensity.

To resolve this and expose the hidden network topology, we must transform our functional matrix using a **row-wise temporal Z-score standardization**.

By subtracting the mean ($\mu$) of each cell's trace and dividing by its standard deviation ($\sigma$), we normalize every neuron's activity to a universal metric: standard deviations of change relative to its own baseline. This forces the downstream clustering algorithm to ignore absolute brightness thresholds and group neurons entirely by the **similarity of their temporal firing patterns**.

---

#### 💻 **Task:**

Use Gemini to write a Python script that applies a row-wise Z-score transformation to `clean_dff` before plotting an optimized hierarchical `clustermap`.

To ensure the script functions perfectly within Google Colab, copy and paste the detailed technical blueprint below into your prompt:

```text
Write a Python script to standardize neural traces and plot an optimized Seaborn clustermap.

The script must fulfill the following architectural requirements:
1. Row-Wise Standardization: Compute the Z-score along the time axis (axis 1) for each individual cell in `clean_dff` using `scipy.stats.zscore` (or manually via NumPy). Save this as `z_matrix`.
2. Hierarchical Clustermap configuration: Pass `z_matrix` directly to `sns.clustermap`.
   - Enable row clustering (`row_cluster=True`) using 'euclidean' distance and 'ward' linkage.
   - Disable column clustering (`col_cluster=False`) to keep the experimental timeline intact.
   - Set the colormap to 'magma' and clip the color limits tightly between `vmin=-1.0` and `vmax=3.0` so resting baseline states remain dark while active deviations stand out clearly.
   - Suppress row text markers (`yticklabels=False`) and set `xticklabels=100` to clean up the axes.
3. Sensory Indicator: Access the underlying heatmap axes via the clustermap object and draw a vertical dashed cyan line at frame 600 to highlight the onset of the dark flashes.

## <font color="blue"> **Transitioning to Network Topology: Symmetrical Correlation Clustermap** </font>




Up to this point, our heatmaps have focused entirely on individual temporal traces. To map the true functional network architecture of the larval brain, we must transition from signal processing into **Network Topology**. We achieve this by computing the **Pairwise Pearson Correlation Matrix**.

The correlation matrix compresses the temporal timeline into a square $(\text{Neurons} \times \text{Neurons})$ matrix, where each entry represents the synchronous coupling strength between a pair of cells. However, because the quiet resting baseline dominates roughly 75% of our recording time, the raw correlation coefficients settle close to zero, rendering a standard matrix uniformly dark. To uncover the hidden network topology without losing full-session information, we apply a **dynamic contrast boost** by lowering our maximum color saturation threshold (`vmax`). By enabling hierarchical clustering symmetrically along **both rows and columns**, cells with identical functional profiles are grouped together, forcing the matrix to organize into high-contrast, square **Functional Blocks** along the diagonal.

---

#### 💻 **Task:**

Use Gemini to write a Python script that computes the pairwise correlation matrix of your neurons and optimizes its visibility using a symmetrical, contrast-boosted clustermap.

To ensure the script functions perfectly within Google Colab, copy and paste the detailed technical blueprint below into your prompt:

```text
Write a Python script to compute a pairwise Pearson correlation matrix and display it using an optimized, contrast-boosted Seaborn clustermap.

The script must fulfill the following architectural requirements:
1. Matrix Calculation: Compute the square $(\text{Neurons} \times \text{Neurons})$ Pearson correlation matrix from `clean_dff` using `np.corrcoef`. Handle any potential flatlined traces gracefully by converting NaN values to 0.0 using `np.nan_to_num`. Save this matrix as `corr_full`.
2. Symmetrical Clustermap Configuration: Pass `corr_full` to `sns.clustermap`.
   - Enable clustering for BOTH rows and columns (`row_cluster=True`, `col_cluster=True`).
   - Use 'euclidean' distance as the metric and 'ward' linkage as the clustering method.
   - Set the colormap to 'magma'.
3. Dynamic Range Contrast Adjustment: To force faint network modules out of the darkness, set the color limits strictly between `vmin=0.0` and a lowered ceiling of `vmax=0.18`.
4. Interface Cleanup: Turn off text tick labels on both axes (`yticklabels=False`, `xticklabels=False`) to ensure a clean, publication-ready diagram. Set the colorbar label to "Functional Coupling Strength (Pearson $R$)".

### **Quantifying Geometric Constraints: Truncated Spatial Decay Models**

Brain networks are physically embedded within the finite geometry of an organism's skull. Because growing and maintaining long-range connections requires significant metabolic energy, spatial network topology typically obeys a strong biophysical constraint: nearby neurons exhibit higher functional coupling, while the magnitude of interaction decays smoothly as a function of anatomical distance.

To formalize this relationship without letting long-range inter-hemispheric projections distort our local metrics, we apply a **spatial truncation threshold** at $d \le 500$ pixels. We evaluate the coupling magnitude by taking the absolute value of our correlation coefficients ($|R|$), which prevents positive and negative anti-phase correlations from cancelling each other out mathematically. We then fit the high-resolution binned means to an exponential decay model:

$$|R(d)| = A e^{-d/\lambda} + C$$

Where $\lambda$ represents the **characteristic length**—the precise spatial scale over which local microcircuit coupling strength drops off.

---

#### 💻 **Task:**

Use Gemini to write a high-performance Python script that applies a geometric truncation filter, groups the surviving links into high-resolution bins, fits an exponential decay function, and overlays the optimized model curve.

To ensure the script functions perfectly within Google Colab, copy and paste the detailed technical blueprint below into your prompt:

```text
Write a Python script to evaluate network spatial decay properties using an exponential fit on truncated pairwise data.

The script must fulfill the following architectural requirements:
1. Pairwise Geometry & Coupling: Calculate the absolute value of your correlation matrix (`np.abs(corr_full)`). Compute the pairwise Euclidean distances between cell centroids using `scipy.spatial.distance.pdist`. Extract the corresponding upper-triangle entries for both matrices to map distances to absolute correlations.
2. Geometric Truncation Mask: Create a boolean filter to retain only cell pairs separated by a distance less than or equal to 500.0 pixels. Apply this mask to drop all long-range pairs from both arrays.
3. High-Resolution Spatial Binning: Define 32 equal-interval distance bins spanning from 0 to 500 pixels using `np.linspace`. Calculate the center of each bin and compute the mean absolute correlation for the pairs within each interval (handle empty bins by inserting `np.nan`).
4. Non-Linear Model Fitting: Define an exponential decay function `exponential_decay(d, A, lam, C)`. Use `scipy.optimize.curve_fit` to fit this model to your valid, non-NaN binned data using reasonable starting guesses (e.g., Amplitude=0.2, Lambda=100.0, Offset=0.05). Extract the optimal parameters and their standard errors.
5. Statistical Evaluation: Calculate the Coefficient of Determination ($R^2$ score) on the binned means to assess the goodness-of-fit. Print a structured text report containing the calculated characteristic length ($\lambda$), the asymptotic floor ($C$), and the final $R^2$ score.
6. Validation Figure: Plot an 11x6 Matplotlib figure displaying:
   - A scatter plot of the empirical binned means (color='darkviolet', size=50).
   - A smooth, continuous red line tracking the optimized exponential decay fit curve.
   - A vertical dotted gray indicator line marking the exact location of the characteristic length ($\lambda$), accompanied by a text label displaying its value.
   - Proper axis labels, a grid (alpha=0.3), limits capped tightly at `plt.xlim(-10, 520)`, and an explicit legend displaying the model's $R^2$ score.